# Research Workflow — deterministic search -> pick -> scrape -> synthesize

Standalone subsystem (kept current so it can be iterated). `%run common.ipynb` first,
then defines Brave search, scraper, the URL picker (LLM-1), the research synthesizer
(LLM-2), `research_workflow`, and the `research` tool that the Draft agent calls.

In [ ]:
import os
# Run from backend/ so `%run common.ipynb` resolves regardless of the kernel's
# launch dir (IDE/Jupyter kernels often start at the workspace root).
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
%run common.ipynb


In [ ]:
# Research subsystem: Brave + scraper + URL-picker (LLM-1) + research synthesizer
# (LLM-2) + research_workflow + `research` tool.
# Assumes `%run common.ipynb` (cell above) loaded client / model / imports.
# --- Brave search ---
BRAVE_RESULT_COUNT = 10
BRAVE_MIN_INTERVAL = 1.1

class BraveRateLimiter:
    def __init__(self, min_interval: float = BRAVE_MIN_INTERVAL):
        self._lock = asyncio.Lock()
        self._min_interval = min_interval
        self._last_call: float = 0.0

    async def __aenter__(self):
        await self._lock.acquire()
        now = time.monotonic()
        gap = now - self._last_call
        if gap < self._min_interval:
            await asyncio.sleep(self._min_interval - gap)
        return self

    async def __aexit__(self, *_):
        self._last_call = time.monotonic()
        self._lock.release()

# Guard create-once: this notebook may be %run more than once (draft.ipynb and
# pipeline.ipynb both pull it in). Re-creating the limiter would reset its timing
# and risk a Brave 429, so create it only if absent.
if "brave_rate_limiter" not in globals():
    brave_rate_limiter = BraveRateLimiter()

async def brave_search(query: str) -> list[dict]:
    url = "https://api.search.brave.com/res/v1/web/search"
    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "X-Subscription-Token": os.environ["BRAVE_API_KEY"],
    }
    params = {"q": query, "count": BRAVE_RESULT_COUNT}
    async with brave_rate_limiter:
        async with httpx.AsyncClient(timeout=15) as client:
            resp = await client.get(url, headers=headers, params=params)
            resp.raise_for_status()
    data = resp.json()
    results = data.get("web", {}).get("results", [])
    return [
        {"title": r.get("title", ""), "url": r.get("url", ""), "description": r.get("description", "")}
        for r in results
    ]

# --- Scraper ---
SCRAPE_CHAR_LIMIT = 3000
HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; SocialPrepBot/1.0)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9",
}

async def _fetch_one(client: httpx.AsyncClient, url: str) -> dict:
    try:
        resp = await client.get(url, headers=HTTP_HEADERS, timeout=10, follow_redirects=True)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = " ".join(soup.get_text(separator=" ").split())
        return {"url": url, "content": text[:SCRAPE_CHAR_LIMIT]}
    except Exception:
        return {"url": url, "content": ""}

async def scrape_urls(urls: list[str]) -> list[dict]:
    """Scrape urls in parallel. Failed/blocked pages are returned with an
    'error' field instead of being silently dropped, so the caller knows a
    retry is futile."""
    async with httpx.AsyncClient() as client:
        results = await asyncio.gather(*[_fetch_one(client, u) for u in urls])
    return [
        r if r["content"] else {"url": r["url"], "error": "blocked or unreachable — retrying will not help"}
        for r in results
    ]

# --- Research schemas ---
class Source(BaseModel):
    label: str = Field(description="Short human-readable source label, e.g. 'Reuters'.")
    url: str
    relevance: str = Field(description="One short sentence on why this source backs the cited claim.")

class ResearchAnswer(BaseModel):
    narrative: str = Field(
        description="Friendly, conversation-ready answer with inline [N] citation markers."
    )
    sources: list[Source] = Field(
        description="One entry per inline citation marker. Do not include uncited sources."
    )

# --- LLM-1: URL picker (no tools, structured output, no query modification) ---
# Judgment only — the pipeline owns search, scraping, and any retry loop.
# pick_urls() guards the output: any URL not copied verbatim from the SERP is
# stripped with a warning (anti-fabrication).
URLS_PER_QUERY = 8

class UrlSelection(BaseModel):
    urls: list[str] = Field(
        description=f"Up to {URLS_PER_QUERY} URLs copied VERBATIM from the search results, "
        "ordered most promising first. Fewer is fine if the results are weak."
    )
    notes: str = Field(
        description="1-2 sentences on the picking rationale: what was prioritized, "
        "what was skipped and why."
    )

URL_PICKER_PROMPT = f"""\
You are a search-result triager for a conversation-prep research pipeline.
Given a search Query, the user's Context and Reason, and a list of web search
results, pick the URLs most likely to contain detailed, conversation-relevant
content. A separate step will scrape your picks and synthesize a report.

For reference, today's date is {TODAY}.

You will receive an input message in this format:
  Query:   <the search string that produced these results>
  Context: <why the user cares>
  Reason:  <why this search matters>
  Search results (JSON): [{{{{"title", "url", "description"}}}}, ...]

Selection criteria:
- Prefer articles over index pages, news stories over aggregators, specific
  content over homepages.
- Use the Context and Reason to judge relevance — pick pages whose content
  the user could actually bring up in conversation.
- Prefer fresh content when titles, URLs, or descriptions indicate dates.
- Judge only from the titles, URLs, and descriptions given.

Hard rules:
- Pick up to {URLS_PER_QUERY} URLs, ordered most promising first.
- Copy each URL VERBATIM from the results. Never invent, modify, or complete a URL.
- Do not modify the query or suggest new searches — pick from what is given.
- If fewer than {URLS_PER_QUERY} results are worth scraping, pick fewer.
  Quality over quantity: a homepage or aggregator that merely lists headlines
  is usually a waste of a scrape.
"""

url_picker_agent = Agent(
    name="UrlPickerAgent",
    instructions=URL_PICKER_PROMPT,
    model=model("google/gemini-3.1-flash-lite"),
    model_settings=model_settings(RESEARCH_EFFORT),
    output_type=UrlSelection,
)

async def pick_urls(query: str, context: str, reason: str, results: list[dict]) -> UrlSelection:
    """Run LLM-1 on a SERP. Strips any URL not present verbatim in the results."""
    input_msg = (
        f"Query: {query}\n"
        f"Context: {context}\n"
        f"Reason: {reason}\n\n"
        f"Search results (JSON):\n{json.dumps(results, ensure_ascii=False)}"
    )
    r = await Runner.run(url_picker_agent, input_msg, max_turns=2)
    sel: UrlSelection = r.final_output

    serp_urls = {item["url"] for item in results}
    kept = []
    for u in sel.urls:
        if u in serp_urls:
            kept.append(u)
        else:
            print(f"[guard] WARNING: invented/modified URL stripped: {u}")
    sel.urls = kept[:URLS_PER_QUERY]
    return sel

# --- LLM-2: research synthesizer (no tools — structurally cannot stall on tool calls) ---
RESEARCH_SYNTHESIZER_PROMPT = f"""\
You are a research synthesizer for a conversation-prep pipeline. You receive
a focused Query, the user's Context and Reason, and raw material scraped from
web pages. Your job is to find the fresh, conversation-ready information in
that material and write the final answer. Capture the essence and ignore any
fluff — a report writer will consume your answer.

For reference, today's date is {TODAY}.

You will receive an input message in this format:
  Query:   <short search string>
  Context: <why the user cares>
  Reason:  <why this search matters>
  followed by blocks of scraped text, each starting with: === PAGE: <url> ===

Write a ResearchAnswer:
- narrative: 4-8 distinct items the user could actually bring up in
  conversation, in flowing prose. For each item, briefly explain what it is
  and why it makes a natural conversation hook given the Context. Place the
  citation marker [N] immediately after the specific sentence that uses that
  source — not at the end of the paragraph, not grouped together at the end.
- sources: one Source entry per citation marker:
    label: short publication name (e.g. 'Reuters', 'BBC', 'Al Jazeera')
    url: copied EXACTLY from a PAGE header in the material
    relevance: one sentence on why this source backs the cited claim

Hard rules:
- Use ONLY the provided material. Do not invent facts.
- Cite only pages whose content appears in the material; copy URLs verbatim
  from the PAGE headers.
- Every [N] marker must appear IMMEDIATELY after the sentence it supports.
- If the material is thin, write fewer items and note the gap honestly in
  the narrative.
"""

research_synthesizer_agent = Agent(
    name="ResearchSynthesizerAgent",
    instructions=RESEARCH_SYNTHESIZER_PROMPT,
    model=model("google/gemini-3.1-flash-lite"),
    model_settings=model_settings(RESEARCH_EFFORT),
    output_type=ResearchAnswer,
)

# --- Deterministic research workflow: search -> pick -> scrape -> (replace once) -> synthesize ---
# The loop lives HERE, in code: exactly one optional replacement round when too
# few pages scrape (the blocked-finance-sites case), then synthesis is forced.
SCRAPE_RETRY_THRESHOLD = 3  # fewer ok pages than this -> one replacement round

async def research_workflow(query: str, context: str, reason: str) -> str:
    """Deterministic replacement for the agentic research agent.
    Returns the same narrative + Sources block string the Draft agent consumes."""
    t0 = time.monotonic()
    serp = await brave_search(query)
    if not serp:
        return f"No usable search results for query='{query}'."

    sel = await pick_urls(query, context, reason, serp)
    scraped = await scrape_urls(sel.urls)
    pages = [r for r in scraped if "content" in r]
    tried = set(sel.urls)
    used_replacement = False

    if len(pages) < SCRAPE_RETRY_THRESHOLD:
        remaining = [r for r in serp if r["url"] not in tried]
        if remaining:
            used_replacement = True
            sel2 = await pick_urls(query, context, reason, remaining)
            scraped2 = await scrape_urls(sel2.urls)
            pages += [r for r in scraped2 if "content" in r]
            tried |= set(sel2.urls)

    if not pages:
        return (f"Research on '{query}' found search results but every page "
                f"failed to scrape. Consider rephrasing the query if you try again.")

    material = "\n\n".join(f"=== PAGE: {p['url']} ===\n{p['content']}" for p in pages)
    synth_input = f"Query: {query}\nContext: {context}\nReason: {reason}\n\n{material}"
    r = await Runner.run(research_synthesizer_agent, synth_input, max_turns=2)
    answer: ResearchAnswer = r.final_output

    # Guard: every cited url must be a scraped page (warn; the draft-pipeline
    # guard also validates downstream at the TalkingPoint level).
    ok_urls = {p["url"] for p in pages}
    for s in answer.sources:
        if s.url not in ok_urls:
            print(f"[guard] WARNING: cited url not in scraped material: {s.url}")

    elapsed = time.monotonic() - t0
    print(f"[workflow] '{query}': picked={len(sel.urls)} scraped_ok={len(pages)} "
          f"replacement_round={used_replacement} sources={len(answer.sources)} "
          f"elapsed={elapsed:.1f}s")

    lines = [answer.narrative, "", "Sources:"]
    for i, src in enumerate(answer.sources, 1):
        lines.append(f"[{i}] {src.label} — {src.url}  ({src.relevance})")
    return "\n".join(lines)

# --- research tool wrapper (what the Draft agent calls) ---
@function_tool
async def research(query: str, context: str, reason: str) -> str:
    """Search the web for fresh information on a focused question.

    Args:
        query: Short Brave search string, e.g. 'Italy recent news'.
        context: Why the user cares, e.g. 'I am meeting a friend who has
            recently been to Italy and want recent news as conversation topics'.
        reason: Why this search matters, e.g. 'Recent events may connect
            to my friend's travel experiences'.
    """
    return await research_workflow(query, context, reason)

print(f"  research workflow ready (search -> pick -> scrape -> synthesize), TODAY = {TODAY}")


In [ ]:
# Demo (standalone only) — gated so `%run` from another notebook stays quiet.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    _out = await research_workflow(
        "stock market recent news 2026",
        "I am meeting a friend who is invested in the stock market.",
        "Recent market news makes good conversation topics.",
    )
    print(_out[:900])
